In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pdb # use 'pdb.set_trace()' where you want to see
from mpl_toolkits.mplot3d import Axes3D
torch.manual_seed(2222)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        n_input = 3
        n_output = 1
        n_nodes = 30

        self.hidden_layer1 = nn.Linear(n_input,n_nodes)
        nn.init.xavier_uniform_(self.hidden_layer1.weight)
        nn.init.normal_(self.hidden_layer1.bias)

        self.hidden_layer2 = nn.Linear(n_nodes,n_nodes)
        nn.init.xavier_uniform_(self.hidden_layer2.weight)
        nn.init.normal_(self.hidden_layer2.bias)

        self.hidden_layer3 = nn.Linear(n_nodes,n_nodes)
        nn.init.xavier_uniform_(self.hidden_layer3.weight)
        nn.init.normal_(self.hidden_layer3.bias)

        self.hidden_layer4 = nn.Linear(n_nodes,n_nodes)
        nn.init.xavier_uniform_(self.hidden_layer4.weight)
        nn.init.normal_(self.hidden_layer4.bias)

        self.hidden_layer5 = nn.Linear(n_nodes,n_nodes)
        nn.init.xavier_uniform_(self.hidden_layer5.weight)
        nn.init.normal_(self.hidden_layer5.bias)

        self.hidden_layer6 = nn.Linear(n_nodes,n_nodes)
        nn.init.xavier_uniform_(self.hidden_layer6.weight)
        nn.init.normal_(self.hidden_layer6.bias)

        self.output_layer = nn.Linear(n_nodes, n_output)
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.normal_(self.output_layer.bias)

    def forward(self, x,y,z):
        inputs = torch.cat([x,y,z],axis=1)

        layer1_out = torch.tanh(self.hidden_layer1(inputs))
        layer2_out = torch.tanh(self.hidden_layer2(layer1_out))
        layer3_out = torch.tanh(self.hidden_layer3(layer2_out))
        layer4_out = torch.tanh(self.hidden_layer4(layer3_out))
        layer5_out = torch.tanh(self.hidden_layer5(layer4_out))
        layer6_out = torch.tanh(self.hidden_layer6(layer5_out))

        output = self.output_layer(layer6_out) ## For regression, no activation is used in output layer
        return output

In [ ]:
def pinnLoss3D(x,y,z, mse, net_u, net_v, net_w):

    u = u_hard(x, y, z)
    v = v_hard(x, y, z)
    w = w_hard(x, y, z)

    # lmbd = E * nu/((1+nu)*(1-2*nu))
    # mu = E/(2*(1+nu))

    #Alu
    # lmbd = 0.504
    # mu = 0.259

    #Steel
    lmbd = 1.076
    mu = 0.779

    #Copper
    # lmbd = 0.854
    # mu = 0.44

    u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
    u_y = torch.autograd.grad(u.sum(), y, create_graph=True)[0]
    u_z = torch.autograd.grad(u.sum(), z, create_graph=True)[0]
    v_x = torch.autograd.grad(v.sum(), x, create_graph=True)[0]
    v_y = torch.autograd.grad(v.sum(), y, create_graph=True)[0]
    v_z = torch.autograd.grad(v.sum(), z, create_graph=True)[0]
    w_x = torch.autograd.grad(w.sum(), x, create_graph=True)[0]
    w_y = torch.autograd.grad(w.sum(), y, create_graph=True)[0]
    w_z = torch.autograd.grad(w.sum(), z, create_graph=True)[0]

    exx = u_x
    eyy = v_y
    ezz = w_z
    exy = 1/2*(u_y + v_x)
    eyz = 1/2*(v_z + w_y)
    ezx = 1/2*(u_z + w_x)

    sxx = lmbd*(exx + eyy + ezz) + 2*mu*exx
    syy = lmbd*(exx + eyy + ezz) + 2*mu*eyy
    szz = lmbd*(exx + eyy + ezz) + 2*mu*ezz
    sxy = 2*mu*exy
    syz = 2*mu*eyz
    szx = 2*mu*ezx

    sxx_x = torch.autograd.grad(sxx.sum(), x, create_graph=True)[0]
    sxy_x = torch.autograd.grad(sxy.sum(), x, create_graph=True)[0]
    szx_x = torch.autograd.grad(szx.sum(), x, create_graph=True)[0]
    sxy_y = torch.autograd.grad(sxy.sum(), y, create_graph=True)[0]
    syy_y = torch.autograd.grad(syy.sum(), y, create_graph=True)[0]
    syz_y = torch.autograd.grad(syz.sum(), y, create_graph=True)[0]
    szx_z = torch.autograd.grad(szx.sum(), z, create_graph=True)[0]
    syz_z = torch.autograd.grad(syz.sum(), z, create_graph=True)[0]
    szz_z = torch.autograd.grad(szz.sum(), z, create_graph=True)[0]

    Fx = (sxx_x + sxy_y + szx_z)
    Fy = (sxy_x + syy_y + syz_z)
    Fz = (szx_x + syz_y + szz_z)

    mse_losspde= mse(Fx, torch.zeros_like(x)) + mse(Fy, torch.zeros_like(x)) + mse(Fz, torch.zeros_like(x))


    return mse_losspde, u, v, w, sxx, syy, szz, sxy, syz, szx


Chia lưới điểm trong miền và biên

In [ ]:
def dataPoints3D(x0, y0, z0, dx, dy, dz, nx, ny, nz, densBnd):
    import numpy as np
    import matplotlib.pyplot as plt

    # ======= Lưới toàn khối =======
    x = np.linspace(x0, x0 + dx, nx)
    y = np.linspace(y0, y0 + dy, ny)
    z = np.linspace(z0, z0 + dz, nz)
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

    # ======= Điểm trong (loại bỏ biên) =======
    Xi = X[1:-1,1:-1,1:-1].ravel().reshape(-1,1)
    Yi = Y[1:-1,1:-1,1:-1].ravel().reshape(-1,1)
    Zi = Z[1:-1,1:-1,1:-1].ravel().reshape(-1,1)

    # ======= Các mặt biên (chia dày hơn) =======
    nx_b = densBnd * nx
    ny_b = densBnd * ny
    nz_b = densBnd * nz

    # Mặt x = x0 và x = x1
    y_x = np.linspace(y0, y0 + dy, ny_b)
    z_x = np.linspace(z0, z0 + dz, nz_b)
    Yx, Zx = np.meshgrid(y_x, z_x, indexing='ij')
    xb0 = np.full_like(Yx, x0).ravel()
    xb1 = np.full_like(Yx, x0 + dx).ravel()
    yb_x = Yx.ravel()
    zb_x = Zx.ravel()

    # Mặt y = y0 và y = y1
    x_y = np.linspace(x0, x0 + dx, nx_b)
    z_y = np.linspace(z0, z0 + dz, nz_b)
    Xy, Zy = np.meshgrid(x_y, z_y, indexing='ij')
    yb0 = np.full_like(Xy, y0).ravel()
    yb1 = np.full_like(Xy, y0 + dy).ravel()
    xb_y = Xy.ravel()
    zb_y = Zy.ravel()

    # Mặt z = z0 và z = z1
    x_z = np.linspace(x0, x0 + dx, nx_b)
    y_z = np.linspace(y0, y0 + dy, ny_b)
    Xz, Yz = np.meshgrid(x_z, y_z, indexing='ij')
    zb0 = np.full_like(Xz, z0).ravel()
    zb1 = np.full_like(Xz, z0 + dz).ravel()
    xb_z = Xz.ravel()
    yb_z = Yz.ravel()

    # ======= Gộp điểm biên =======
    Xb = np.concatenate([xb0, xb1, xb_y, xb_y, xb_z, xb_z]).reshape(-1,1)
    Yb = np.concatenate([yb_x, yb_x, yb0, yb1, yb_z, yb_z]).reshape(-1,1)
    Zb = np.concatenate([zb_x, zb_x, zb_y, zb_y, zb0, zb1]).reshape(-1,1)

    # ======= Gộp tất cả điểm lại =======
    X_all = np.vstack([Xi, Xb])
    Y_all = np.vstack([Yi, Yb])
    Z_all = np.vstack([Zi, Zb])

    return X_all, Y_all, Z_all


Xác định các biên

In [ ]:
def boundaryPointsBox(x, y, z, x0, y0, z0, x1, y1, z1, atol=1e-5):
    device = x.device
    dtype = x.dtype

    Lx = x1 - x0
    Ly = y1 - y0
    Lz = z1 - z0

    # Ép kiểu và đưa về tensor
    x0_tensor = torch.tensor(x0, device=device, dtype=dtype)
    x1_tensor = torch.tensor(x1, device=device, dtype=dtype)
    y0_tensor = torch.tensor(y0, device=device, dtype=dtype)
    y1_tensor = torch.tensor(y1, device=device, dtype=dtype)
    z0_tensor = torch.tensor(z0, device=device, dtype=dtype)
    z1_tensor = torch.tensor(z1, device=device, dtype=dtype)

    # Biên x = x0
    idx_x0 = torch.where(torch.isclose(x, x0_tensor, atol=atol))[0]

    # Biên x = x1 chia Load và Free
    idx_x1 = torch.where(torch.isclose(x, x1_tensor, atol=atol))[0]

    # Biên y
    idx_y0 = torch.where(torch.isclose(y, y0_tensor, atol=atol))[0]
    idx_y1 = torch.where(torch.isclose(y, y1_tensor, atol=atol))[0]

    # Biên z
    idx_z0 = torch.where(torch.isclose(z, z0_tensor, atol=atol))[0]
    idx_z1 = torch.where(torch.isclose(z, z1_tensor, atol=atol))[0]

    return idx_x0, idx_x1, idx_y0, idx_y1, idx_z0, idx_z1


Loss biên

In [ ]:
def boundaryLoss3D(f, x, y, z, x0, y0, z0, x1, y1, z1, mse, net_u, net_v, net_w):

    # ---- Biên x = x1 - phần load ----
    xR = x[idx_x1].view(-1,1)
    yR = y[idx_x1].view(-1,1)
    zR = z[idx_x1].view(-1,1)
    _, _, _, _, sxx_R, syy_R, szz_R, sxy_R, sxz_R, syz_R = pinnLoss3D(xR, yR, zR, mse, net_u, net_v, net_w)

    # ---- Các biên tự do (Neumann = 0) ----
    def free_neumann(idx):
        xB = x[idx].view(-1,1)
        yB = y[idx].view(-1,1)
        zB = z[idx].view(-1,1)
        _, _, _, _, sxx, syy, szz, sxy, sxz, syz = pinnLoss3D(xB, yB, zB, mse, net_u, net_v, net_w)
        return sxx, syy, szz, sxy, sxz, syz

    sxx_y0, syy_y0, szz_y0, sxy_y0, sxz_y0, syz_y0 = free_neumann(idx_y0)
    sxx_y1, syy_y1, szz_y1, sxy_y1, sxz_y1, syz_y1 = free_neumann(idx_y1)
    sxx_z0, syy_z0, szz_z0, sxy_z0, sxz_z0, syz_z0 = free_neumann(idx_z0)
    sxx_z1, syy_z1, szz_z1, sxy_z1, sxz_z1, syz_z1 = free_neumann(idx_z1)

    # ==== MSE loss ====

    # Neumann tại x = x1 (load) — chỉ có syz ≠ 0 (giả sử lực theo y)
    loss_R_sxx = mse(sxx_R, torch.zeros_like(sxx_R))
    loss_R_sxy = mse(sxy_R, torch.zeros_like(sxy_R))
    loss_R_sxz = mse(sxz_R, torch.zeros_like(sxz_R))
    loss_R_syz = mse(syz_R, g * torch.ones_like(syz_R))

    # Các biên tự do khác
    loss_y0 = sum(mse(t, torch.zeros_like(t)) for t in [sxx_y0, syy_y0, szz_y0, sxy_y0, sxz_y0, syz_y0])
    loss_y1 = sum(mse(t, torch.zeros_like(t)) for t in [sxx_y1, syy_y1, szz_y1, sxy_y1, sxz_y1, syz_y1])
    loss_z0 = sum(mse(t, torch.zeros_like(t)) for t in [sxx_z0, syy_z0, szz_z0, sxy_z0, sxz_z0, syz_z0])
    loss_z1 = sum(mse(t, torch.zeros_like(t)) for t in [sxx_z1, syy_z1, szz_z1, sxy_z1, sxz_z1, syz_z1])

    # Tổng
    loss_bc = (
          loss_R_sxx + loss_R_sxy + loss_R_sxz + loss_R_syz
        + loss_y0 + loss_y1 + loss_z0 + loss_z1
    )

    return loss_bc/28.0


In [ ]:
x0, y0, z0 = 0, 0, 0
x1, y1, z1 = 1.0, 0.1, 0.1
nx, ny, nz = 12, 6, 6
densBnd = 10
g = -1
xnp, ynp, znp = dataPoints3D(x0, y0, z0, x1, y1, z1, nx, ny, nz, densBnd)


In [ ]:
# Định nghĩa λ trainable (ban đầu = 1.0)
lambda_pde = nn.Parameter(torch.tensor(1.0, dtype=torch.float32, requires_grad=True).to(device))
lambda_bc  = nn.Parameter(torch.tensor(1.0, dtype=torch.float32, requires_grad=True).to(device))

# Chuyển dữ liệu sang torch và bật requires_grad để tự động đạo hàm
x = torch.tensor(xnp, dtype=torch.float32, requires_grad=True).to(device)
y = torch.tensor(ynp, dtype=torch.float32, requires_grad=True).to(device)
z = torch.tensor(znp, dtype=torch.float32, requires_grad=True).to(device)

idx_x0, idx_x1, idx_y0, idx_y1, idx_z0, idx_z1 = boundaryPointsBox(x, y, z, x0, y0, z0, x1, y1, z1)

# Khởi tạo mạng
net_u = Net().to(device)
net_v = Net().to(device)
net_w = Net().to(device)

def u_hard(x, y, z):
    return x * net_u(x, y, z)

def v_hard(x, y, z):
    return x * net_v(x, y, z)  # v(x=0,y,z) = 0

def w_hard(x, y, z):
    return x * net_w(x, y, z)  # w(x=0,y,z) = 0

# Cài đặt training
mse = torch.nn.MSELoss()
optimizer_net = torch.optim.Adam(
    list(net_u.parameters()) + list(net_v.parameters()) + list(net_w.parameters()),
    lr=1e-3
)
optimizer_lambda = torch.optim.Adam([lambda_pde, lambda_bc], lr=5e-4)

optimizer_lbfgs = torch.optim.LBFGS(
    list(net_u.parameters()) + list(net_v.parameters()) + list(net_w.parameters()),
    lr=1.0,
    max_iter=5000,
    tolerance_grad=1e-9,
    tolerance_change=1e-12,
    history_size=100,
    line_search_fn='strong_wolfe'
)

history = np.zeros((0, 2))


In [ ]:
def plot_loss(loss_adam, loss_lbfgs=None, ylabel='Training Loss'):
    epochs_adam = len(loss_adam)
    x_adam = np.arange(1, epochs_adam + 1)

    plt.figure(figsize=(8,5))
    plt.plot(x_adam, loss_adam, color='blue', lw=1.5, label='Adam')

    if loss_lbfgs is not None:
        x_lbfgs = np.arange(epochs_adam + 1, epochs_adam + 1 + len(loss_lbfgs))
        plt.plot(x_lbfgs, loss_lbfgs, color='red', lw=1.5, label='L-BFGS')

    plt.xlabel('Iteration')
    plt.ylabel(ylabel)
    plt.yscale('log')
    plt.gca().yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2e'))
    plt.grid(True, which='both', linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig('loss_history.png')
    plt.show()

# Huấn luyện PINN 3D
loss_history = []
loss_history_lbfgs = []
lambda_history = []

# Giới hạn lambda để tránh bùng nổ
lambda_min, lambda_max = 1e-2, 100.0

warmup_epochs = 100
num_epochs_adam = 7000
# alpha = 0.12
alpha = 0.3

L0_pde, L0_bc = None, None   # loss ban đầu cho GradNorm

start_time = time.time()

for epoch in range(num_epochs_adam):
    t0 = time.time()
    # ================== Update network ==================
    optimizer_net.zero_grad()

    lam_pde = torch.clamp(torch.relu(lambda_pde), lambda_min, lambda_max)
    lam_bc  = torch.clamp(torch.relu(lambda_bc),  lambda_min, lambda_max)

    # Forward lần 1
    mse_losspde = pinnLoss3D(x, y, z, mse, net_u, net_v, net_w)[0]
    mse_lossbc  = boundaryLoss3D(g, x, y, z,
                                 x0, y0, z0, x1, y1, z1,
                                 mse, net_u, net_v, net_w)

    if epoch == 0:
        L0_pde = mse_losspde.detach()
        L0_bc  = mse_lossbc.detach()

    loss = lam_pde * mse_losspde + lam_bc * mse_lossbc
    loss.backward()
    optimizer_net.step()

    # ================== Update λ với GradNorm ==================
    if epoch >= warmup_epochs and (epoch % 100 == 0):
        mse_losspde2 = pinnLoss3D(x, y, z, mse, net_u, net_v, net_w)[0]
        mse_lossbc2  = boundaryLoss3D(g, x, y, z,
                                      x0, y0, z0, x1, y1, z1,
                                      mse, net_u, net_v, net_w)

        lam_pde2 = torch.clamp(torch.relu(lambda_pde), lambda_min, lambda_max)
        lam_bc2  = torch.clamp(torch.relu(lambda_bc),  lambda_min, lambda_max)

        W = [p for p in net_u.parameters()] + [p for p in net_v.parameters()] + [p for p in net_w.parameters()]

        G_pde_list = torch.autograd.grad(lam_pde2 * mse_losspde2, W,
                                    retain_graph=True, create_graph=True, allow_unused=True)
        G_bc_list  = torch.autograd.grad(lam_bc2  * mse_lossbc2,  W,
                                    retain_graph=True, create_graph=True, allow_unused=True)

        # Norm theo GradNorm: trung bình norm từng layer
        G_pde = torch.stack([g.norm() for g in G_pde_list if g is not None]).mean()
        G_bc = torch.stack([g.norm() for g in G_bc_list if g is not None]).mean()

        # W = list(net_u.parameters())[0]

        # G_pde = torch.autograd.grad(lam_pde2 * mse_losspde2, W,
        #                             retain_graph=True, create_graph=True)[0].norm()
        # G_bc  = torch.autograd.grad(lam_bc2  * mse_lossbc2, W,
        #                             retain_graph=True, create_graph=True)[0].norm()

        G_avg = 0.5 * (G_pde + G_bc)

        r_pde = (mse_losspde2.detach() / L0_pde) / (
                 0.5 * ((mse_losspde2.detach()/L0_pde) + (mse_lossbc2.detach()/L0_bc)))
        r_bc  = (mse_lossbc2.detach() / L0_bc) / (
                 0.5 * ((mse_losspde2.detach()/L0_pde) + (mse_lossbc2.detach()/L0_bc)))

        target_G_pde = G_avg * (r_pde ** alpha)
        target_G_bc  = G_avg * (r_bc ** alpha)

        L_grad = (torch.abs(G_pde - target_G_pde) +
                  torch.abs(G_bc  - target_G_bc))

        optimizer_lambda.zero_grad()
        L_grad.backward()
        optimizer_lambda.step()

    # ================== Logging ==================
    loss_history.append(loss.item())
    lambda_history.append((lam_pde.item(), lam_bc.item()))

    t1 = time.time() - t0

    if (epoch + 1) % 1000 == 0 or epoch == 0:
        print(f"[Epoch {epoch+1:4d}] Total Loss: {loss.item():.4e}")
        print(f"    PDE Loss: {mse_losspde:.4e} | BC Loss: {mse_lossbc:.4e}")
        print(f"    lam_pde={lam_pde.item():.3f}, lam_bc={lam_bc.item():.3f}, time={t1:.2f}s")

adam_time = time.time() - start_time
print(f"===== KẾT THÚC ADAM, Tổng thời gian: {adam_time:.4f}s =====")

In [ ]:
start_lbfgs = time.time()
lbfgs_epoch = 0  # đếm số vòng lặp


def closure():
    global lbfgs_epoch
    optimizer_lbfgs.zero_grad()

    # freeze lamda sau khi Adam
    lam_pde_val = torch.clamp(torch.relu(lambda_pde.detach()), lambda_min, lambda_max).to(device)
    lam_bc_val  = torch.clamp(torch.relu(lambda_bc.detach()),  lambda_min, lambda_max).to(device)


    mse_losspde = pinnLoss3D(x, y, z, mse, net_u, net_v, net_w)[0]
    mse_lossbc  = boundaryLoss3D(g, x, y, z,
                                 x0, y0, z0, x1, y1, z1,
                                 mse, net_u, net_v, net_w)
    loss = lam_pde_val * mse_losspde + lam_bc_val * mse_lossbc
    loss.backward()

    lbfgs_epoch += 1

    # Lưu loss cho L-BFGS
    loss_history_lbfgs.append(loss.item())

    print(f"[LBFGS step {lbfgs_epoch}] "
              f"Total={loss.item():.6e}, "
              f"PDE={mse_losspde.item():.6e}, "
              f"BC={mse_lossbc.item():.6e}")


    return loss

optimizer_lbfgs.step(closure)


# In kết quả cuối cùng
lbfgs_time = time.time() - start_lbfgs
print(f"===== KẾT THÚC L-BFGS, Tổng thời gian: {lbfgs_time:.4f}s =====")
# Tổng thời gian huấn luyện Adam + L-BFGS
total_time = time.time() - start_time
print(f"===== Tổng thời gian: {total_time:.4f}s =====")

In [ ]:
def plot_loss(loss_adam, loss_lbfgs=None, ylabel='Training Loss'):
    epochs_adam = len(loss_adam)
    x_adam = np.arange(1, epochs_adam + 1)

    plt.figure(figsize=(8,5))
    plt.plot(x_adam, loss_adam, color='blue', lw=1.5, label='Adam')

    if loss_lbfgs is not None:
        # Trục x liên tục nối liền Adam và L-BFGS
        x_lbfgs = np.arange(epochs_adam + 1, epochs_adam + 1 + len(loss_lbfgs))
        plt.plot(x_lbfgs, loss_lbfgs, color='red', lw=1.5, label='L-BFGS')

    plt.xlabel('Iteration')
    plt.ylabel(ylabel)
    plt.yscale('log')
    # plt.gca().yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2e'))
    # Ví dụ đặt các tick trên trục y
    # yticks = [0.01, 0.1, 1]
    # plt.yticks(yticks, [r'$10^{-2}$', r'$10^{-1}$', r'$10^{0}$'])

    plt.xticks([0, 4000, 8000, 10000])

    plt.tick_params(axis='both', which='minor', length=0)
    # plt.grid(True, which='both', linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig('loss_history.png')
    plt.show()

In [ ]:
# Vẽ biểu đồ
plot_loss(loss_history, loss_lbfgs=loss_history_lbfgs, ylabel='Training Loss')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Load dữ liệu FreeFEM++
data = np.loadtxt('/content/drive/MyDrive/PINNs/freefem_solution_3D.txt')
x_ff, y_ff, z_ff = data[:,0], data[:,1], data[:,2]
u1_ff, u2_ff, u3_ff = data[:,3], data[:,4], data[:,5]

# 2. Chuyển sang tensor nếu PINN dùng PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x_ff_t = torch.tensor(x_ff, dtype=torch.float32, device=device).unsqueeze(-1)
y_ff_t = torch.tensor(y_ff, dtype=torch.float32, device=device).unsqueeze(-1)
z_ff_t = torch.tensor(z_ff, dtype=torch.float32, device=device).unsqueeze(-1)
# Chuyển dữ liệu chính xác sang tensor
u1_ff_t = torch.tensor(u1_ff, dtype=torch.float32, device=device).unsqueeze(-1)
u2_ff_t = torch.tensor(u2_ff, dtype=torch.float32, device=device).unsqueeze(-1)
u3_ff_t = torch.tensor(u3_ff, dtype=torch.float32, device=device).unsqueeze(-1)

# 3. Dự đoán từ PINN
with torch.no_grad():
    u1_pred = net_u(x_ff_t, y_ff_t, z_ff_t)  # net_u trả u1
    u2_pred = net_v(x_ff_t, y_ff_t, z_ff_t)  # net_v trả u2
    u3_pred = net_w(x_ff_t, y_ff_t, z_ff_t)  # net_w trả u3
# Ghép thành tensor (N,3)
pred = torch.cat([u1_pred, u2_pred, u3_pred], dim=1)
exact = torch.cat([u1_ff_t, u2_ff_t, u3_ff_t], dim=1)

# 4. Tính MSE bằng PyTorch
mse = torch.nn.MSELoss()
mse_val = mse(pred, exact).item()
print(f"MSE Error (u,v,w): {mse_val:.4e}")

In [ ]:
def get_displacement_field_3D(x, y, z, net_u, net_v, net_w):
    """
    Trả về toạ độ gốc và vector dịch chuyển từ các mạng học được.
    """
    with torch.no_grad():
        u = net_u(x, y, z)
        v = net_v(x, y, z)
        w = net_w(x, y, z)

    x_np = x.detach().cpu().numpy()
    y_np = y.detach().cpu().numpy()
    z_np = z.detach().cpu().numpy()
    u_np = u.detach().cpu().numpy()
    v_np = v.detach().cpu().numpy()
    w_np = w.detach().cpu().numpy()

    return x_np, y_np, z_np, u_np, v_np, w_np


# === Kích thước khối hộp ban đầu ===
dx, dy, dz = 1.0, 0.1, 0.1

scale_factor = 1

# === Tính dịch chuyển và toạ độ sau biến dạng ===
x_np, y_np, z_np, u_np, v_np, w_np = get_displacement_field_3D(x, y, z, net_u, net_v, net_w)
x_def = x_np + u_np*scale_factor
y_def = y_np + v_np*scale_factor
z_def = z_np + w_np*scale_factor

# === Vẽ hình dạng ban đầu và sau biến dạng ===
fig = plt.figure(figsize=(20,12))
ax = fig.add_subplot(111, projection='3d')

# Miền ban đầu
# ax.scatter(x_np, y_np, z_np, s=1, color='blue', alpha=0.2, label='Original')

# Miền sau biến dạng
ax.scatter(x_def, y_def, z_def, s=1, color='blue', alpha=1)

x_span = x_def.max() - x_def.min()
y_span = y_def.max() - y_def.min()
z_span = z_def.max() - z_def.min()
ax.set_box_aspect([x_span, y_span, z_span])


ax.view_init(elev=30, azim=35)

plt.axis('off')
# ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
def get_displacement_field_3D(x, y, z, net_u, net_v, net_w):
    with torch.no_grad():
        u = net_u(x, y, z)
        v = net_v(x, y, z)
        w = net_w(x, y, z)

    return (
        x.detach().cpu().numpy(),
        y.detach().cpu().numpy(),
        z.detach().cpu().numpy(),
        u.detach().cpu().numpy(),
        v.detach().cpu().numpy(),
        w.detach().cpu().numpy()
    )

# === Kích thước khối hộp ban đầu ===
dx, dy, dz = 1.0, 0.1, 0.1

# === Lấy toạ độ và dịch chuyển ===
x_np, y_np, z_np, u_np, v_np, w_np = get_displacement_field_3D(x, y, z, net_u, net_v, net_w)

# === Downsample cho dễ quan sát ===
stride = 20  # Hiển thị mỗi 20 điểm
x_plot = x_np[::stride]
y_plot = y_np[::stride]
z_plot = z_np[::stride]
u_plot = u_np[::stride]
v_plot = v_np[::stride]
w_plot = w_np[::stride]

# === Scale vector dịch chuyển nếu quá nhỏ ===
scale_factor = 20  # Tăng nếu vector ngắn
u_plot_scaled = u_plot * scale_factor
v_plot_scaled = v_plot * scale_factor
w_plot_scaled = w_plot * scale_factor

# === Vẽ quiver 3D ===
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

ax.quiver(
    x_plot, y_plot, z_plot,
    u_plot_scaled, v_plot_scaled, w_plot_scaled,
    length=0.01, normalize=False, color='blue', linewidth=0.2
)

# === Đặt tỷ lệ trục đúng ===
ax.set_box_aspect([dx, dy, dz])

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Displacement Vector Field in 3D (Downsampled & Scaled)")
plt.tight_layout()
plt.show()
